# Analyze Tracks and Select a Person

Inspect the latest resolved run without rerunning model inference, select the person to render, and verify individual frames.

## 1. Configuration

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY

# Set to a run path/name to override the master, environment, or runs/latest.json.
RUN_DIRECTORY_OVERRIDE = None
if globals().get("PERSON_TRACKER_MASTER", False):
    RUN_DIRECTORY_OVERRIDE = globals().get("RUN_DIRECTORY")

print(f"Project root: {PROJECT_ROOT}")
print(f"Configured run override: {RUN_DIRECTORY_OVERRIDE}")

## 2. Load the latest resolved run

In [ ]:
import json
import sys
from collections import Counter, defaultdict

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from person_tracker.io import read_nth_frame
from person_tracker.storage import load_observation_run, resolve_run_directory

RUN_DIRECTORY = resolve_run_directory(
    PROJECT_ROOT / "runs", stage="resolved", explicit=RUN_DIRECTORY_OVERRIDE,
)
required_files = [
    "manifest.json", "tracks.jsonl", "face_samples.jsonl",
    "face_embeddings.npy", "identities.jsonl",
]
missing_files = [name for name in required_files if not (RUN_DIRECTORY / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Resolved run is incomplete; missing: {missing_files}")

manifest, tracking_history, face_samples = load_observation_run(
    RUN_DIRECTORY, load_face_crops=True
)
identity_history = defaultdict(dict)
with (RUN_DIRECTORY / "identities.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        if line.strip():
            record = json.loads(line)
            frame_no = int(record.pop("frame"))
            track_id = int(record.pop("track_id"))
            identity_history[frame_no][track_id] = record
identity_history = dict(identity_history)

summary_path = RUN_DIRECTORY / "identity_summary.json"
events_path = RUN_DIRECTORY / "identity_events.json"
identity_summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
switch_events = json.loads(events_path.read_text()) if events_path.exists() else {}
INPUT_VIDEO = Path(manifest["source_video"])
fps = float(manifest["fps"])
start_frame = int(manifest["start_frame"])
end_frame = int(manifest["end_frame"])

print(f"Run: {RUN_DIRECTORY}")
print(f"Source: {INPUT_VIDEO}")
print(f"Frames: {start_frame}:{end_frame} ({(end_frame-start_frame)/fps:.2f}s)")
print(f"Tracks: {len({track['track_id'] for tracks in tracking_history.values() for track in tracks})}")
print(f"Face samples: {len(face_samples):,}")
print(f"Resolved people: {identity_summary.get('person_count', 'unknown')}")

## 3. Compare tracks and people

In [ ]:
track_stats = defaultdict(lambda: {
    "frames": [], "confidences": [], "people": Counter(), "sources": Counter()
})
person_stats = defaultdict(lambda: {
    "frames": set(), "tracks": set(), "confidences": [], "sources": Counter()
})
for frame_no, tracks in tracking_history.items():
    identities = identity_history.get(frame_no, {})
    for track in tracks:
        track_id = int(track["track_id"])
        tstats = track_stats[track_id]
        tstats["frames"].append(frame_no)
        tstats["confidences"].append(float(track["confidence"]))
        identity = identities.get(track_id)
        if identity:
            person_id = int(identity["person_id"])
            source = identity.get("source", "unknown")
            tstats["people"][person_id] += 1
            tstats["sources"][source] += 1
            pstats = person_stats[person_id]
            pstats["frames"].add(frame_no)
            pstats["tracks"].add(track_id)
            pstats["confidences"].append(float(identity.get("confidence", 0.0)))
            pstats["sources"][source] += 1

faces_by_track = Counter(sample.track_id for sample in face_samples)
faces_by_person = defaultdict(list)
for sample in face_samples:
    identity = identity_history.get(sample.frame_no, {}).get(sample.track_id)
    if identity:
        faces_by_person[int(identity["person_id"])].append(sample)

track_table = pd.DataFrame([
    {
        "Track": track_id, "First": min(stats["frames"]),
        "Last": max(stats["frames"]), "Frames": len(stats["frames"]),
        "Duration (s)": len(stats["frames"]) / fps,
        "Mean detection": np.mean(stats["confidences"]),
        "People": ", ".join(f"P{pid} ({count})" for pid, count in stats["people"].most_common()),
        "Faces": faces_by_track[track_id],
    }
    for track_id, stats in sorted(track_stats.items())
])
person_table = pd.DataFrame([
    {
        "Person": person_id, "First": min(stats["frames"]),
        "Last": max(stats["frames"]), "Frames": len(stats["frames"]),
        "Duration (s)": len(stats["frames"]) / fps,
        "Tracks": sorted(stats["tracks"]), "Track count": len(stats["tracks"]),
        "Faces": len(faces_by_person.get(person_id, [])),
        "Mean identity confidence": np.mean(stats["confidences"]),
    }
    for person_id, stats in sorted(person_stats.items())
])
display(HTML("<h3>People</h3>"), person_table.style.format({
    "Duration (s)": "{:.2f}", "Mean identity confidence": "{:.3f}"
}))
display(HTML("<h3>Tracks</h3>"), track_table.style.format({
    "Duration (s)": "{:.2f}", "Mean detection": "{:.3f}"
}))

## 4. Inspect visibility and identity switches

In [ ]:
fig, ax = plt.subplots(figsize=(15, max(4, 0.55 * len(person_stats))))
for person_id in sorted(person_stats):
    frames = sorted(person_stats[person_id]["frames"])
    ax.scatter([(frame-start_frame)/fps for frame in frames], [person_id]*len(frames), s=8)
ax.set(title="Resolved person visibility", xlabel="Seconds from run start", ylabel="Person ID")
ax.set_yticks(sorted(person_stats))
ax.grid(True, axis="x", alpha=0.25)
plt.show()

event_table = pd.DataFrame([
    {"Track": int(track_id), **event}
    for track_id, events in switch_events.items() for event in events
])
display(HTML("<h3>Identity-switch boundaries</h3>"))
display(event_table if not event_table.empty else HTML("<i>No switch events recorded.</i>"))

## 5. Select the person to render

The selected person is saved inside this run and loaded automatically by notebook 04.

In [ ]:
from person_tracker.ui import create_person_selection_widget

SELECTION_PATH = RUN_DIRECTORY / "selected_person.json"
selector, selection_state = create_person_selection_widget(
    faces_by_person, person_stats, selection_path=SELECTION_PATH, fps=fps,
)
display(selector)

## 6. Inspect annotated frames

Move through the analyzed range and optionally isolate one resolved person.

In [ ]:
person_options = [("All people", 0)] + [(f"Person {pid}", pid) for pid in sorted(person_stats)]

def show_annotated_frame(frame_no, person_id):
    frame = read_nth_frame(INPUT_VIDEO, frame_no)
    identities = identity_history.get(frame_no, {})
    shown = 0
    for track in tracking_history.get(frame_no, []):
        track_id = int(track["track_id"])
        identity = identities.get(track_id)
        resolved_person = int(identity["person_id"]) if identity else None
        if person_id and resolved_person != person_id:
            continue
        x1, y1, x2, y2 = map(int, track["bbox"])
        color = (0, 255, 0) if identity else (255, 0, 0)
        label = f"T{track_id} | P{resolved_person if resolved_person is not None else '--'}"
        if identity:
            label += f" | {float(identity.get('confidence', 0)):.0%}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 5)
        cv2.putText(frame, label, (x1, max(35, y1-12)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)
        shown += 1
    plt.figure(figsize=(16, 9))
    plt.imshow(frame)
    plt.title(f"Frame {frame_no} | {(frame_no-start_frame)/fps:.2f}s | boxes shown: {shown}")
    plt.axis("off")
    plt.show()

frame_slider = widgets.IntSlider(
    value=start_frame, min=start_frame, max=max(start_frame, end_frame-1), step=1,
    description="Frame", continuous_update=False, layout=widgets.Layout(width="80%"),
)
person_dropdown = widgets.Dropdown(options=person_options, value=0, description="Show")
viewer = widgets.interactive_output(
    show_annotated_frame, {"frame_no": frame_slider, "person_id": person_dropdown}
)
display(widgets.VBox([person_dropdown, frame_slider]), viewer)